# 03 - Gold: métricas de negócio e automação com Streams e Tasks

**Objetivo:**
1. Criar a camada **Gold**: tabelas que respondem perguntas do time de marketing.
2. Simular a **chegada de pedidos novos** e ver o pipeline reagir sozinho.

| Tabela Gold | Pergunta que responde |
|---|---|
| `RECEITA_DIARIA_CANAL` | Quanto vendemos por dia em cada canal? Qual o ticket médio? |
| `RFM_CLIENTES` | Quem são os clientes campeões e quem está sumindo? |
| `FUNIL_CAMPANHAS` | Qual canal de campanha converte mais? |

**Tempo estimado:** 30 minutos

In [ ]:
%%sql -r dataframe_1
-- Contexto da sessao: quem sou eu, qual motor e qual gaveta vou usar.
-- Rode esta celula SEMPRE que abrir o notebook.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ESPM_WH;
USE DATABASE ESPM_HOL;
USE SCHEMA GOLD;

## Parte A - Gold com Dynamic Tables

**Ticket médio** = receita ÷ número de pedidos. É uma das métricas mais olhadas no varejo.

### EXERCICIO 1 - Complete a métrica de ticket médio

Complete os `???` na célula abaixo.

**Dica:** ticket médio = receita dividida pela quantidade de pedidos. As duas expressões já estão nas linhas de cima.

**Travou?** Abra o Cortex Code e peça:
> Complete o cálculo de ticket_medio (receita / quantidade de pedidos) com 2 casas decimais

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE DYNAMIC TABLE GOLD.RECEITA_DIARIA_CANAL
  TARGET_LAG = '1 hour'
  WAREHOUSE = ESPM_WH
  COMMENT = 'Receita e ticket medio por dia e canal'
AS
SELECT
    data_pedido,
    canal_venda,
    COUNT(*)                                       AS pedidos,
    SUM(valor_total)                               AS receita,
    ROUND(??? / ???, 2)                            AS ticket_medio
FROM SILVER.PEDIDOS
WHERE pedido_valido
GROUP BY data_pedido, canal_venda;

In [ ]:
%%sql -r dataframe_3
-- RFM: Recencia (ha quantos dias comprou), Frequencia (quantas vezes), Valor (quanto gastou)
-- Data de referencia fixa (31/10/2026) para todo mundo ter o mesmo resultado na aula
CREATE OR REPLACE DYNAMIC TABLE GOLD.RFM_CLIENTES
  TARGET_LAG = '1 hour'
  WAREHOUSE = ESPM_WH
  COMMENT = 'Segmentacao RFM de clientes'
AS
SELECT
    p.cliente_id,
    c.nome,
    c.cidade,
    DATEDIFF('day', MAX(p.data_pedido), '2026-10-31'::DATE)  AS recencia_dias,
    COUNT(*)                                                 AS frequencia,
    SUM(p.valor_total)                                       AS valor_total,
    CASE
      WHEN COUNT(*) >= 3 AND DATEDIFF('day', MAX(p.data_pedido), '2026-10-31'::DATE) <= 90 THEN 'CAMPEOES'
      WHEN COUNT(*) >= 2                                                                    THEN 'FIEIS'
      WHEN DATEDIFF('day', MAX(p.data_pedido), '2026-10-31'::DATE) <= 60                  THEN 'NOVOS'
      ELSE 'EM_RISCO'
    END                                                      AS segmento
FROM SILVER.PEDIDOS p
LEFT JOIN SILVER.CLIENTES c ON c.cliente_id = p.cliente_id
WHERE p.pedido_valido
GROUP BY p.cliente_id, c.nome, c.cidade;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE DYNAMIC TABLE GOLD.FUNIL_CAMPANHAS
  TARGET_LAG = '1 hour'
  WAREHOUSE = ESPM_WH
  COMMENT = 'Funil de campanhas por canal'
AS
SELECT
    ca.nome_canal,
    ca.tipo_canal,
    COUNT(*)                                           AS disparos,
    COUNT_IF(d.abriu)                                  AS aberturas,
    COUNT_IF(d.clicou)                                 AS cliques,
    COUNT(p.pedido_id)                                 AS conversoes,
    COALESCE(SUM(p.valor_total), 0)                    AS receita_atribuida,
    ROUND(COUNT_IF(d.abriu)  / COUNT(*) * 100, 1)      AS taxa_abertura_pct,
    ROUND(COUNT_IF(d.clicou) / NULLIF(COUNT_IF(d.abriu), 0) * 100, 1) AS taxa_clique_pct
FROM SILVER.CAMPANHAS_DISPAROS d
JOIN SILVER.CANAIS_CAMPANHA ca ON ca.canal_id = d.canal_id
LEFT JOIN SILVER.PEDIDOS p ON p.pedido_id = d.pedido_convertido_id AND p.pedido_valido
GROUP BY ca.nome_canal, ca.tipo_canal;

### Checkpoint Gold (antes da carga nova)

In [ ]:
%%sql -r dataframe_5
-- Esperado: pedidos 267 | receita 248639.35
SELECT SUM(pedidos) AS pedidos, SUM(receita) AS receita
FROM GOLD.RECEITA_DIARIA_CANAL;

In [ ]:
%%sql -r dataframe_6
-- Pergunta de negocio: qual canal vende mais e com qual ticket?
SELECT canal_venda, SUM(pedidos) AS pedidos, SUM(receita) AS receita,
       ROUND(SUM(receita) / SUM(pedidos), 2) AS ticket_medio
FROM GOLD.RECEITA_DIARIA_CANAL
GROUP BY canal_venda
ORDER BY receita DESC;

## Parte B - Streams e Tasks: o "despertador" do pipeline

**Cenário:** é Black Friday. Chegam 30 pedidos novos e o time de CRM quer ser **avisado de todo pedido acima de R$ 1.000** para mandar um mimo VIP.

- **Stream** = um "marca-texto" que destaca só as linhas **novas** de uma tabela.
- **Task** = um "despertador" que executa SQL num horário ou depois de outra task.

```mermaid
flowchart LR
  T1[Task CARREGA_PEDIDOS_NOVOS] -->|COPY INTO| BP[BRONZE.PEDIDOS]
  BP -->|marca linhas novas| ST[Stream]
  ST --> T2[Task ALERTA_PEDIDOS_GRANDES]
  T2 --> AL[GOLD.ALERTAS_PEDIDOS_GRANDES]
  BP -->|Dynamic Tables| G[GOLD atualizada]
```

In [ ]:
%%sql -r dataframe_7
-- O marca-texto: a partir de agora, toda linha nova em BRONZE.PEDIDOS fica registrada aqui
CREATE OR REPLACE STREAM ESPM_HOL.BRONZE.PEDIDOS_STREAM
  ON TABLE ESPM_HOL.BRONZE.PEDIDOS
  APPEND_ONLY = TRUE;

-- Tabela que o CRM vai consultar
CREATE OR REPLACE TABLE ESPM_HOL.GOLD.ALERTAS_PEDIDOS_GRANDES (
  pedido_id    VARCHAR,
  cliente_id   VARCHAR,
  canal_venda  VARCHAR,
  valor_total  NUMBER(12,2),
  detectado_em TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Antes da carga: o stream esta vazio (esperado 0)
SELECT COUNT(*) AS linhas_novas FROM ESPM_HOL.BRONZE.PEDIDOS_STREAM;

In [ ]:
%%sql -r dataframe_8
-- Task 1 (raiz): busca o arquivo de pedidos novos no GitHub e carrega na Bronze
CREATE OR REPLACE TASK ESPM_HOL.BRONZE.CARREGA_PEDIDOS_NOVOS
  WAREHOUSE = ESPM_WH
  SCHEDULE = '60 MINUTE'
  COMMENT = 'Carga incremental de pedidos'
AS
EXECUTE IMMEDIATE $$
BEGIN
  ALTER GIT REPOSITORY ESPM_HOL.PUBLIC.HOL_REPO FETCH;
  COPY FILES INTO @ESPM_HOL.BRONZE.ARQUIVOS/incremental/
    FROM @ESPM_HOL.PUBLIC.HOL_REPO/branches/main/incremental/;
  COPY INTO ESPM_HOL.BRONZE.PEDIDOS
    FROM @ESPM_HOL.BRONZE.ARQUIVOS/incremental/
    FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
    INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);
END;
$$;

-- Task 2 (filha): roda depois da Task 1, mas SO se o stream tiver novidades
CREATE OR REPLACE TASK ESPM_HOL.BRONZE.ALERTA_PEDIDOS_GRANDES
  WAREHOUSE = ESPM_WH
  AFTER ESPM_HOL.BRONZE.CARREGA_PEDIDOS_NOVOS
  WHEN SYSTEM$STREAM_HAS_DATA('ESPM_HOL.BRONZE.PEDIDOS_STREAM')
AS
INSERT INTO ESPM_HOL.GOLD.ALERTAS_PEDIDOS_GRANDES (pedido_id, cliente_id, canal_venda, valor_total)
SELECT pedido_id, cliente_id, canal_venda, TRY_TO_NUMBER(valor_total, 12, 2)
FROM ESPM_HOL.BRONZE.PEDIDOS_STREAM
WHERE TRY_TO_NUMBER(valor_total, 12, 2) >= 1000;

Tasks nascem **pausadas** (SUSPENDED), por segurança. A task filha precisa estar ligada (`RESUME`) para rodar quando a raiz terminar. A raiz vamos disparar manualmente com `EXECUTE TASK`.

### EXERCICIO 2 - Ligue a task filha e dispare a raiz

Complete os `???` na célula abaixo.

**Dica:** o contrário de SUSPEND é...

**Travou?** Abra o Cortex Code e peça:
> Como eu ativo uma task suspensa no Snowflake?

In [ ]:
%%sql -r dataframe_9
ALTER TASK ESPM_HOL.BRONZE.ALERTA_PEDIDOS_GRANDES ???;

-- Dispara agora, sem esperar o agendamento
EXECUTE TASK ESPM_HOL.BRONZE.CARREGA_PEDIDOS_NOVOS;

**Aguarde ~30-60 segundos** e rode a célula abaixo até as duas tasks aparecerem como `SUCCEEDED`.

In [ ]:
%%sql -r dataframe_10
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(ESPM_HOL.INFORMATION_SCHEMA.TASK_HISTORY(
       SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
WHERE database_name = 'ESPM_HOL'
ORDER BY scheduled_time DESC
LIMIT 10;

## Parte C - A Gold acompanha sozinha

A Gold tem `TARGET_LAG = '1 hour'`. Para não esperar uma hora, pedimos um **refresh agora**. O Snowflake atualiza também a Silver que está no meio do caminho.

In [ ]:
%%sql -r dataframe_11
ALTER DYNAMIC TABLE ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL REFRESH;
ALTER DYNAMIC TABLE ESPM_HOL.GOLD.RFM_CLIENTES REFRESH;
ALTER DYNAMIC TABLE ESPM_HOL.GOLD.FUNIL_CAMPANHAS REFRESH;

### Checkpoint final

In [ ]:
%%sql -r dataframe_12
-- Esperado: pedidos_bronze 330 | pedidos_gold 293 | receita 273770.12 | alertas 10
SELECT
  (SELECT COUNT(*)     FROM ESPM_HOL.BRONZE.PEDIDOS)                 AS pedidos_bronze,
  (SELECT SUM(pedidos) FROM ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL)      AS pedidos_gold,
  (SELECT SUM(receita) FROM ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL)      AS receita,
  (SELECT COUNT(*)     FROM ESPM_HOL.GOLD.ALERTAS_PEDIDOS_GRANDES)   AS alertas;

In [ ]:
%%sql -r dataframe_13
SELECT * FROM ESPM_HOL.GOLD.ALERTAS_PEDIDOS_GRANDES ORDER BY valor_total DESC;

## Higiene de custo

Desligamos a task filha para ela não ficar esperando à toa. A raiz nunca foi ligada (`RESUME`), só disparada manualmente.

In [ ]:
%%sql -r dataframe_14
ALTER TASK ESPM_HOL.BRONZE.ALERTA_PEDIDOS_GRANDES SUSPEND;

## Resumo

- **Gold** = dado pronto para decisão: receita, ticket, RFM e funil.
- **Stream + Task** = reação a eventos (pedido grande vira alerta).
- **Dynamic Tables** = a cadeia Bronze, Silver e Gold se recalcula sem código extra.

**Próximo:** `04_snowpark` - fazer o mesmo tipo de trabalho, mas em Python.